In [ ]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer, IndexToString
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, ClusteringEvaluator

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("ColabPySpark4") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [ ]:
# --- 2. DEFINE YOUR CATEGORIES ---
# This ensures the model maps these specific strings to indices 0-29
category_list = [
    'Larceny Theft', 'Weapons Offense', 'Motor Vehicle Theft', 'Burglary', 'Courtesy Report',
    'Traffic Collision', 'Vehicle Impounded', 'Vandalism', 'Lost Property', 'Suspicious Occ',
    'Embezzlement', 'Fraud', 'Family Offense', 'Stolen Property', 'Forgery And Counterfeiting',
    'Warrant', 'Traffic Violation Arrest', 'Fire Report', 'Drug Offense', 'Disorderly Conduct',
    'Sex Offense', 'Recovered Vehicle', 'Missing Person', 'Robbery', 'Assault', 'Arson',
    'Offences Against The Family And Children', 'Prostitution', 'Civil Sidewalks', 'Weapons Carrying Etc'
]

In [ ]:
# Load the uploaded CSV
# Note: Adjust 'sep' if it's commas, but your snippet looks like tabs/special quoting
df = spark.read.csv('train.csv', header=True, inferSchema=True)

# Show the first few rows to verify
df.show(5)

+------------------------------------------------------------------------------------------------+--------------------+--------------------+----+----+----+
|\tDate\tCategory\tDescription\tDay of Week\tPdDistrict\tResolution\tAddress\tLongitude\tLatitude|                 _c1|                 _c2| _c3| _c4| _c5|
+------------------------------------------------------------------------------------------------+--------------------+--------------------+----+----+----+
|                                                                            0\t1580712300.0\t...|                NULL|                NULL|NULL|NULL|NULL|
|                                                                            1\t1580672700.0\t...| Possession with ...| Receiving\tMonda...|NULL|NULL|NULL|
|                                                                            6\t1579104060.0\t...|                NULL|                NULL|NULL|NULL|NULL|
|                                                               

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, Tokenizer, StopWordsRemover,
                                HashingTF, IDF, VectorAssembler)
from pyspark.ml.classification import LogisticRegression, LinearSVC, GBTClassifier, OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import time
import sys

# --- 3. DATA LOADING & CLEANING ---
def clean_crime_data(path):
    df_raw = spark.read.text(path)

# Regex logic:
# 1. Remove the starting quote (")
# 2. Remove trailing quotes and any extra commas (",,,,,)
# 3. Split the remaining string by tabs (\t)

    # Regex to handle your specific CSV format (tabs inside quotes with trailing commas)
    cleaned = df_raw.select(F.split(F.regexp_replace(F.col("value"), r'^"|",*$', ""), "\t").alias("cols"))

    # Define column names
    cols_names = ["ID", "Timestamp", "Category", "Description", "DayOfWeek", "PdDistrict", "Resolution", "Address", "X", "Y"]
    return cleaned.select([F.col("cols")[i].alias(cols_names[i]) for i in range(len(cols_names))]) \
                  .filter(F.col("Category") != "Category").dropna()

def main():
    # 1. Initialize Spark Session
    spark = SparkSession.builder \
        .appName("CrimeCategoryClassification") \
        .getOrCreate()

    # Optional: Set log level to INFO to see progress in EMR stderr logs
    spark.sparkContext.setLogLevel("INFO")

    # 2. Load Data (Update these paths to your S3 bucket or local EMR path)
    # Example: "s3://my-bucket/train.csv"
    train_path = "train.csv"
    test_path = "test.csv"

    # Load as raw text
    try:
        df_raw_train = spark.read.text(train_path)
        df_raw_test = spark.read.text(test_path)
    except Exception as e:
        print(f"Error loading files: {e}")
        sys.exit(1)

    # Clean using the method
    df_train = clean_crime_data(df_raw_train)
    df_test = clean_crime_data(df_raw_test)


    # 3. Data Sanitization (Prevent Target Leakage)
    # Replaces the Category name inside the Description with a neutral placeholder
    df_train = df_train.withColumn("Description",
        F.regexp_replace(F.col("Description"), F.col("Category"), "CRIME_TYPE"))

    # 4. Feature Engineering Pipeline Stages
    l_indexer = StringIndexer(inputCol="Category", outputCol="label", handleInvalid="keep")
    tokenizer = Tokenizer(inputCol="Description", outputCol="words")
    remover = StopWordsRemover(inputCol="words", outputCol="filtered")
    hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=5000)
    idf = IDF(inputCol="rawFeatures", outputCol="features")
    assembler = VectorAssembler(inputCols=["features"], outputCol="final_features")

    feature_stages = [l_indexer, tokenizer, remover, hashingTF, idf, assembler]

    # 5. Define the Models
    # Logistic Regression
    lr = LogisticRegression(maxIter=20, regParam=0.1, labelCol="label", featuresCol="final_features")

    # Support Vector Machine (Wrapped in OvR for Multiclass)
    lsvc = LinearSVC(maxIter=10, regParam=0.1)
    svm_ovr = OneVsRest(classifier=lsvc, labelCol="label", featuresCol="final_features")

    # Gradient Boosted Trees (Wrapped in OvR for Multiclass)
    # Keeping maxIter low as GBT-OvR is very computationally expensive on EMR
    gbt = GBTClassifier(maxIter=5, maxDepth=5)
    gbt_ovr = OneVsRest(classifier=gbt, labelCol="label", featuresCol="final_features")

    models = {
        "Logistic Regression": lr,
        "Support Vector Machine (OvR)": svm_ovr,
        "Gradient Boosted Trees (OvR)": gbt_ovr
    }

    # 6. Evaluation Execution
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

    # Cache training data to speed up iterative algorithms (LR, SVM, GBT)
    df_train.cache()

    print("\n" + "="*50)
    print("STARTING MODEL EVALUATION")
    print("="*50)

    results_report = []

    for name, algo in models.items():
        print(f"\n[TRAINING] Starting {name}...")
        start_time = time.time()

        # Build Pipeline
        pipeline = Pipeline(stages=feature_stages + [algo])

        # Fit Model
        model = pipeline.fit(df_train)

        # Predict
        predictions = model.transform(df_test)

        # Evaluate
        accuracy = evaluator.evaluate(predictions)
        duration_mins = (time.time() - start_time) / 60

        result_str = f"{name:<30}: {accuracy:.2%} (Duration: {duration_mins:.2f} mins)"
        results_report.append(result_str)
        print(f"[FINISHED] {result_str}")

    # 7. Final Summary Output
    print("\n" + "="*50)
    print("FINAL ACCURACY REPORT")
    print("="*50)
    for line in results_report:
        print(line)
    print("="*50 + "\n")

    spark.stop()

if __name__ == "__main__":
    main()

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `_get_object_id` is not supported.

In [ ]:
# --- 3. DATA LOADING & CLEANING ---
def clean_crime_data(path):
    df_raw = spark.read.text(path)

# Regex logic:
# 1. Remove the starting quote (")
# 2. Remove trailing quotes and any extra commas (",,,,,)
# 3. Split the remaining string by tabs (\t)

    # Regex to handle your specific CSV format (tabs inside quotes with trailing commas)
    cleaned = df_raw.select(F.split(F.regexp_replace(F.col("value"), r'^"|",*$', ""), "\t").alias("cols"))

    # Define column names
    cols_names = ["ID", "Timestamp", "Category", "Description", "DayOfWeek", "PdDistrict", "Resolution", "Address", "X", "Y"]
    return cleaned.select([F.col("cols")[i].alias(cols_names[i]) for i in range(len(cols_names))]) \
                  .filter(F.col("Category") != "Category").dropna()

In [ ]:
df_train = clean_crime_data("train.csv")
df_test = clean_crime_data("test.csv")

In [ ]:
df_train.show()

+---+------------+--------------------+--------------------+---------+----------+--------------------+--------------------+-------------------+------------------+
| ID|   Timestamp|            Category|         Description|DayOfWeek|PdDistrict|          Resolution|             Address|                  X|                 Y|
+---+------------+--------------------+--------------------+---------+----------+--------------------+--------------------+-------------------+------------------+
|  0|1580712300.0|      Missing Person|        Found Person|   Monday|   Taraval|      Open or Active|20TH AVE \ WINSTO...|-122.47603947349434| 37.72694991292525|
|  1|1580672700.0|     Stolen Property|Stolen Property",...|   Monday|   Mission|Cite or Arrest Adult|24TH ST \ SHOTWEL...|-122.41517229045436| 37.75243964438968|
|  6|1579104060.0|Offences Against ...|Elder Adult or De...| Thursday|   Mission|           Unfounded|18TH ST \ SANCHEZ ST| -122.4305735690657| 37.76115547557552|
| 11|1578646800.0|    

In [ ]:
df_train = df_train.withColumn("Description",
    F.regexp_replace(F.col("Description"), F.col("Category"), "CRIME_TYPE"))

In [ ]:
df_train.show(5)

+---+------------+--------------------+--------------------+---------+----------+--------------------+--------------------+-------------------+-----------------+
| ID|   Timestamp|            Category|         Description|DayOfWeek|PdDistrict|          Resolution|             Address|                  X|                Y|
+---+------------+--------------------+--------------------+---------+----------+--------------------+--------------------+-------------------+-----------------+
|  0|1580712300.0|      Missing Person|        Found Person|   Monday|   Taraval|      Open or Active|20TH AVE \ WINSTO...|-122.47603947349434|37.72694991292525|
|  1|1580672700.0|     Stolen Property|CRIME_TYPE", Poss...|   Monday|   Mission|Cite or Arrest Adult|24TH ST \ SHOTWEL...|-122.41517229045436|37.75243964438968|
|  6|1579104060.0|Offences Against ...|Elder Adult or De...| Thursday|   Mission|           Unfounded|18TH ST \ SANCHEZ ST| -122.4305735690657|37.76115547557552|
| 11|1578646800.0|       Los

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
import time

# --- 1. SETUP THE FEATURE PREPROCESSING ---
# These stages remain the same for both the search and the final model
l_indexer = StringIndexer(inputCol="Category", outputCol="label", handleInvalid="keep")
tokenizer = Tokenizer(inputCol="Description", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=5000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
assembler = VectorAssembler(inputCols=["features"], outputCol="final_features")

feature_stages = [l_indexer, tokenizer, remover, hashingTF, idf, assembler]

# --- 2. AUTOMATED K-SELECTION (The "Search" Phase) ---
# We use a 10% sample to find natural groupings fast on EMR
search_df = df_train.sample(False, 0.1, seed=42).cache()
feature_pipeline_model = Pipeline(stages=feature_stages).fit(search_df)
processed_search = feature_pipeline_model.transform(search_df).cache()

best_k = 2
max_silhouette = -1.0
evaluator = ClusteringEvaluator(featuresCol="final_features", metricName="silhouette")

print("Searching for natural groupings (k=2 to 20)...")
for k in range(2, 21):
    kmeans = KMeans(k=k, featuresCol="final_features", seed=1)
    model = kmeans.fit(processed_search)
    predictions = model.transform(processed_search)

    silhouette = evaluator.evaluate(predictions)
    print(f"Testing k={k:2d}: Silhouette = {silhouette:.4f}")

    if silhouette > max_silhouette:
        max_silhouette = silhouette
        best_k = k

print(f"\nOPTIMUM K FOUND: {best_k}")

# --- 3. FINAL PRODUCTION PIPELINE ---
# Now we plug the 'best_k' into the final pipeline and train on 100% of the data
final_kmeans = KMeans(k=best_k, featuresCol="final_features", seed=1, predictionCol="natural_cluster")
final_pipeline = Pipeline(stages=feature_stages + [final_kmeans])

print(f"Training final pipeline on full dataset with k={best_k}...")
final_model = final_pipeline.fit(train_df)

# --- 4. EXECUTION & RESULTS ---
final_predictions = final_model.transform(test_df)
final_predictions.select("Description", "Category", "natural_cluster").show(10)

Searching for natural groupings (k=2 to 20)...
Testing k= 2: Silhouette = -0.1563
Testing k= 3: Silhouette = -0.3045
Testing k= 4: Silhouette = -0.2897
Testing k= 5: Silhouette = -0.2654
Testing k= 6: Silhouette = -0.2269
Testing k= 7: Silhouette = -0.2624
Testing k= 8: Silhouette = -0.2263
Testing k= 9: Silhouette = -0.2780
Testing k=10: Silhouette = -0.2775
Testing k=11: Silhouette = -0.1004
Testing k=12: Silhouette = -0.0828
Testing k=13: Silhouette = -0.0781
Testing k=14: Silhouette = -0.0849
Testing k=15: Silhouette = -0.0604
Testing k=16: Silhouette = -0.0513
Testing k=17: Silhouette = -0.0311
Testing k=18: Silhouette = -0.0233
Testing k=19: Silhouette = -0.0152
Testing k=20: Silhouette = 0.0029

OPTIMUM K FOUND: 20
Training final pipeline on full dataset with k=20...


NameError: name 'train_df' is not defined

In [ ]:
from pyspark.ml.feature import VectorAssembler

# 1. Index the Labels
l_indexer = StringIndexer(inputCol="Category", outputCol="label", handleInvalid="keep")

# 2. Text Processing
tokenizer = Tokenizer(inputCol="Description", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=5000)
idf = IDF(inputCol="rawFeatures", outputCol="features")

# 3. Vector Assembler (Crucial Step)
# This combines your text features into the format Spark models expect
assembler = VectorAssembler(inputCols=["features"], outputCol="final_features")

# Update your models to use "final_features" instead of "features"
lr = LogisticRegression(maxIter=10, regParam=0.3, labelCol="label", featuresCol="final_features")
rf = RandomForestClassifier(numTrees=10, labelCol="label", featuresCol="final_features")
km = KMeans(k=30, seed=1, featuresCol="final_features", predictionCol="prediction")

# Update the feature stages
feature_stages = [l_indexer, tokenizer, remover, hashingTF, idf, assembler]

In [ ]:
# --- 4. MODEL EXECUTION & DETAILED METRICS ---
models = {
    "Logistic Regression": LogisticRegression(labelCol="label", featuresCol="final_features", maxIter=15),
    "Random Forest": RandomForestClassifier(labelCol="label", featuresCol="final_features", numTrees=20),
    "K-Means Clustering": KMeans(k=30, featuresCol="final_features", predictionCol="prediction", seed=1)
}

report_lines = ["MODEL EVALUATION REPORT", "="*40]

for name, algo in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline(stages=feature_stages + [algo])
    model = pipeline.fit(df_train)
    preds = model.transform(df_test)

    report_lines.append(f"\n[ALGORITHM: {name}]")

    if "Clustering" in name:
        evaluator = ClusteringEvaluator(featuresCol="final_features", predictionCol="prediction")
        score = evaluator.evaluate(preds)
        report_lines.append(f"Silhouette Score: {score:.4f}")
    else:
        # Multiclass Evaluator
        eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

        # Gathering multiple metrics
        acc  = eval.evaluate(preds, {eval.metricName: "accuracy"})
        f1   = eval.evaluate(preds, {eval.metricName: "f1"})
        prec = eval.evaluate(preds, {eval.metricName: "weightedPrecision"})
        rec  = eval.evaluate(preds, {eval.metricName: "weightedRecall"})

        report_lines.append(f"Accuracy:  {acc:.2%}")
        report_lines.append(f"Precision: {prec:.2%}")
        report_lines.append(f"Recall:    {rec:.2%}")
        report_lines.append(f"F1 Score:  {f1:.2%}")

        # AUC ROC (Requires probability column from classification models)
        try:
            auc = eval.evaluate(preds, {eval.metricName: "areaUnderROC"})
            report_lines.append(f"AUC ROC:   {auc:.4f}")
        except:
            report_lines.append("AUC ROC:   Not applicable for this model version")

Training Logistic Regression...
Training Random Forest...
Training K-Means Clustering...


In [ ]:
# --- 5. EXPORTING THE REPORT ---
final_report = "\n".join(report_lines)
print(final_report)

MODEL EVALUATION REPORT

[ALGORITHM: Logistic Regression]
Accuracy:  93.47%
Precision: 89.56%
Recall:    93.47%
F1 Score:  91.35%
AUC ROC:   Not applicable for this model version

[ALGORITHM: Random Forest]
Accuracy:  40.27%
Precision: 16.22%
Recall:    40.27%
F1 Score:  23.13%
AUC ROC:   Not applicable for this model version

[ALGORITHM: K-Means Clustering]
Silhouette Score: 0.2182


In [ ]:
# Ensure no IDs overlap
overlap = df_train.join(df_test, "ID").count()
print(f"Number of overlapping rows: {overlap}") # This SHOULD be 0

Number of overlapping rows: 0


In [ ]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"Test Accuracy: {accuracy:.2%}")

# Show some sample predictions
predictions.select("Description", "Category", "prediction").show(10)

IllegalArgumentException: [FIELD_NOT_FOUND] No such struct field `prediction` in `Description`, `Category`, `label`, `words`, `filtered`, `rawFeatures`, `features`. SQLSTATE: 42704

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# 1. Define the K-Means Model
# k=30 because you have 30 crime categories
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=5, seed=42)

# 2. Build the Pipeline
# We use the same 'feature_stages' from the previous step (Tokenizer, HashingTF, IDF)
kmeans_pipeline = Pipeline(stages=feature_stages + [kmeans])

# 3. Train the Model
# Clustering usually uses the whole dataset or just the training set
kmeans_model = kmeans_pipeline.fit(df_train_filtered)

# 4. Make Predictions (Assigning descriptions to clusters)
clustered_data = kmeans_model.transform(df_test_filtered)

# 5. Evaluate the Clustering
# We use the Silhouette score (measures how similar an object is to its own cluster)
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="cluster", metricName="silhouette")
silhouette = evaluator.evaluate(clustered_data)

print(f"Silhouette with squared euclidean distance = {silhouette:.4f}")

# Show the results: See which Description fell into which Cluster
clustered_data.select("Description", "Category", "cluster").show(10)